#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700">

### Installs and Imports

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [ ]:
#installing dependencies for browser-use (an alternative to skyvern)
%pip install browser-use playwright
%pip install -U langchain-google-genai
%pip install -U browser-use
# %playwright install chromium

### Import API keys from .env file
#### (Note: This is for local use. Colab imports keys differently)

In [6]:
import os
from dotenv import load_dotenv
import platform
import sys


#Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
#added this macro so we don't have to manually comment/uncommment os.chdir() every time.

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")
    print("Running on macOS")
else:
    print("Not running on macOS, current directory:", os.getcwd())


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

Running on macOS


INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.schemas
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.tables
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.types
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.constraints
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.defaults
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.comments


Key check: FOUND


### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [ ]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [7]:
import asyncio
import nest_asyncio
import os
from google import genai
from IPython.display import display, Markdown

nest_asyncio.apply()
client_genai = genai.Client(api_key=gemini_key)

def parse_resume(file_path, prompt="Please parse this resume and return its contents in a clear, readable format. Use sections with headers (like Contact, Summary, Experience, Education, Skills) and plain text or bullet points — no JSON or code blocks."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    Caches the result so Gemini is only called once per resume.
    """
    cache_path = file_path.replace(".pdf", "_cached.json")

    # delete old cache if it exists to force re-parse with new format
    # if os.path.exists(cache_path):
    #     os.remove(cache_path)
    #     print(f"Cleared old cache at {cache_path}.")

    if os.path.exists(cache_path):
        print(f"Using cached resume data from {cache_path}...")
        with open(cache_path, "r") as f:
            return f.read()

    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

    # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    # save to cache so we don't call Gemini again
    with open(cache_path, "w") as f:
        f.write(response.text)
    print(f"Resume parsed and cached to {cache_path}")

    return response.text

# call the function
result = parse_resume("content/jakes-resume.pdf")
display(Markdown(f"<div style='font-size: 14px'>{result}</div>"))

Using cached resume data from content/jakes-resume_cached.json...


<div style='font-size: 14px'>Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
        *   Explored ways to visualize GitHub collaboration in a classroom setting.
*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   September 2018 - Present
        *   Communicate with managers to set up campus computers used on campus.
        *   Assess and troubleshoot computer problems brought by students, faculty, and staff.
        *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus.
*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
        *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*.
        *   Developed a game in Java to test the generated dungeons.
        *   Contributed 50K+ lines of code to an established codebase via Git.
        *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable.
        *   Wrote an 8-page paper and gave multiple presentations on-campus.
        *   Presented virtually to the World Conference on Computational Intelligence.

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
        *   Developed a full-stack web application using Flask serving a REST API with React as the frontend.
        *   Implemented GitHub OAuth to get data from user's repositories.
        *   Visualized GitHub data to show collaboration.
        *   Used Celery and Redis for asynchronous tasks.
*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
        *   Developed a Minecraft server plugin to entertain kids during free time for a previous job.
        *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review.
        *   Implemented continuous delivery using TravisCI to build the plugin upon a new release.
        *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin.

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

## Team Report #2 (03/31) - Created human in the loop for resume info verification, set up ChromaDB, and opened a window of the job website (dummy url) and filled in a few fields.
### (Replaced Skyvern with Browser_Use). This code will not run on Colab, only locally.

<img src="AI_AGENT_WORKFLOW_REPORT_2.png" width="800">

### Developed a human in the loop where agent takes in the extracted info and ask the user if the info is correct. Until the user is satisifed, the agent will keep updating the extracted info based on user's feedback.

In [8]:
from IPython.display import display, Markdown

def verify_resume_info(extracted_info):
    current_info = extracted_info
    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        display(Markdown(f"<div style='font-size: 16px'>{current_info}</div>"))
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()
        print(f"Is this information correct? (y/n): {user_confirm}")

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            break
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()
            print(f"What needs to be corrected or added? Please describe: {correction}")

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}
The user wants the following correction or addition:
{correction}
Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")
    return current_info


# call the function to store the extracted info
result = parse_resume("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

Using cached resume data from content/jakes-resume_cached.json...

EXTRACTED RESUME INFO:


<div style='font-size: 16px'>Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
        *   Explored ways to visualize GitHub collaboration in a classroom setting.
*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   September 2018 - Present
        *   Communicate with managers to set up campus computers used on campus.
        *   Assess and troubleshoot computer problems brought by students, faculty, and staff.
        *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus.
*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
        *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*.
        *   Developed a game in Java to test the generated dungeons.
        *   Contributed 50K+ lines of code to an established codebase via Git.
        *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable.
        *   Wrote an 8-page paper and gave multiple presentations on-campus.
        *   Presented virtually to the World Conference on Computational Intelligence.

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
        *   Developed a full-stack web application using Flask serving a REST API with React as the frontend.
        *   Implemented GitHub OAuth to get data from user's repositories.
        *   Visualized GitHub data to show collaboration.
        *   Used Celery and Redis for asynchronous tasks.
*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
        *   Developed a Minecraft server plugin to entertain kids during free time for a previous job.
        *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review.
        *   Implemented continuous delivery using TravisCI to build the plugin upon a new release.
        *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin.

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

Is this information correct? (y/n): y

Great! Moving forward with this information.


### Setting up ChromaDB which will store the correct parsed info of user's resume

In [9]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path):
    doc_id = hashlib.md5(file_path.encode()).hexdigest()
    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path}]
    )
    print(f"Resume stored! ID: {doc_id}")
    return doc_id

# store the verified resume
doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf")
print("Resume stored successfully!")

CHROMA_HOST: api.trychroma.com
CHROMA_TENANT: 7eab8b90-a03c-4bc3-9704-e5e4f8a3f82b
CHROMA_DATABASE: phil-job-agent
CHROMA_API_KEY: FOUND
Connected to ChromaDB! Total resumes stored: 1
Resume stored! ID: b7a0358df030937c049bb9754117f552
Resume stored successfully!


## Team Report #3 (04/14) - TBD

### Created a popup window of the job website that user will feed into Phil

In [ ]:
import asyncio
import os
from browser_use import Agent, Browser, BrowserProfile
from browser_use.llm import ChatOpenAI  # ← browser-use's own wrapper, NOT langchain_openai

# --- Configuration ---
OPENROUTER_API_KEY = os.getenv("OPENROUTER_KEY")
TARGET_URL = "https://demo.applitools.com/"
MODEL = "anthropic/claude-3.5-sonnet"
MODEL = "anthropic/claude-sonnet-4.5"
MODEL = "google/gemini-2.5-flash-lite"

async def main():
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        model=MODEL,
    )

    browser = Browser(
        browser_profile=BrowserProfile(
            headless=False,
            disable_security=False,
            keep_alive=True,
        )
    )

    task = (
        f"Go to {TARGET_URL}. "
        f"Type 'test_user' into the username field. "
        f"Type 'password123' into the password field. "
        f"STOP. Do NOT click any button. Do NOT press Enter. Do NOT submit the form. "
        f"Return DONE when both fields are filled."
    )

    agent = Agent(
        task=task,
        llm=llm,
        browser=browser,
        use_vision=True,
        max_actions=10,
    )

    result = await agent.run()
    print("Result:", result)

    # await browser.close()


# Jupyter-compatible (no asyncio.run)
await main()

INFO     [Agent] 🔗 Found URL in task: https://demo.applitools.com/, adding as initial action...
INFO     [Agent] 🎯 Task: Go to https://demo.applitools.com/. Type 'test_user' into the username field. Type 'password123' into the password field. STOP. Do NOT click any button. Do NOT press Enter. Do NOT submit the form. Return DONE when both fields are filled.
INFO     [Agent] Starting a browser-use agent with version 0.12.6, with provider=openai and model=google/gemini-2.5-flash-lite
INFO     [Agent]   ▶️   navigate: url: https://demo.applitools.com/, new_tab: False
INFO     [tools] 🔗 Navigated to https://demo.applitools.com/
INFO     [Agent] 

INFO     [Agent] 📍 Step 1:
INFO     [Agent]   👍 Eval: Successfully navigated to the demo.applitools.com page. Verdict: Success
INFO     [Agent]   🧠 Memory: Navigated to https://demo.applitools.com/. The page is still loading skeleton content. The username and password fields are visible.
INFO     [Agent]   🎯 Next goal: Input 'test_user' into the us

Result: AgentHistoryList(all_results=[ActionResult(is_done=False, success=None, judgement=None, error=None, attachments=None, images=None, long_term_memory='Found initial url and automatically loaded it. Navigated to https://demo.applitools.com/', extracted_content='🔗 Navigated to https://demo.applitools.com/', include_extracted_content_only_once=False, metadata=None, include_in_memory=False), ActionResult(is_done=False, success=None, judgement=None, error=None, attachments=None, images=None, long_term_memory="Typed 'test_user'", extracted_content="Typed 'test_user'", include_extracted_content_only_once=False, metadata={'input_x': 756.0, 'input_y': 407.1875}, include_in_memory=False), ActionResult(is_done=False, success=None, judgement=None, error=None, attachments=None, images=None, long_term_memory="Typed 'password123'", extracted_content="Typed 'password123'", include_extracted_content_only_once=False, metadata={'input_x': 756.0, 'input_y': 486.375}, include_in_memory=False), Action

WARNING  [BrowserSession] [SessionManager] Agent focus target 37F0B723... detached! Current focus: None (already cleared). Auto-recovering by switching to another target...
WARNING  [BrowserSession] [SessionManager] No tabs remain! Creating new tab for agent...
WARNING  [BrowserSession] Cannot navigate - browser not connected
ERROR    [BrowserSession] [SessionManager] ❌ Error during agent_focus recovery: RuntimeError: {'code': -32000, 'message': 'Failed to open a new tab'}
WARNING  [BrowserSession] 🔌 CDP WebSocket message handler exited unexpectedly (connection closed)
WARNING  [BrowserSession] 🔄 WebSocket reconnection attempt 1/3...
INFO     [BrowserSession] [SessionManager] Cleared all owned data (targets, sessions, mappings)
WARNING  [BrowserSession] 🔄 Reconnection attempt 1 failed: ConnectionRefusedError: [Errno 61] Connect call failed ('127.0.0.1', 50385)
WARNING  [BrowserSession] 🔄 WebSocket reconnection attempt 2/3...
WARNING  [BrowserSession] 🔄 Reconnection attempt 2 failed: Co

### A function to write to a CSV file with details of the job application. If a CSV file does not exist, it will create one.

In [15]:
import csv
import os

def save_application_history(url, company_name, time_applied, job_title):
    file_exists = os.path.isfile("application_history.csv")
    with open("application_history.csv", mode="a", newline="") as csvfile:
        fieldnames = ["url", "company_name", "time_applied", "job_title"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()

        writer.writerow({"url": url, "company_name": company_name, "time_applied": time_applied, "job_title": job_title})
    print("Application history updated.")


In [16]:
# Prompts user for additional info (demographics, work authorization, etc.)
# and stores it in ChromaDB for future use.
def collect_and_store_additional_info():
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")

    # Check if info already exists
    existing = additional_collection.get(ids=["user_info"])
    if existing["documents"]:
        print("✅ Additional info already stored in ChromaDB.")
        update = input("Would you like to update it? (y/n): ").strip().lower()
        if update != "y":
            return existing["documents"][0]

    print("\n📋 Please provide the following additional info (press Enter to skip any field):")

    gender = input("Gender (e.g. Male, Female, Non-binary, Prefer not to say): ").strip()
    race = input("Race/Ethnicity (e.g. Asian, White, Hispanic, Black, Prefer not to say): ").strip()
    veteran_status = input("Veteran status (e.g. Not a veteran, Veteran, Prefer not to say): ").strip()
    disability_status = input("Disability status (e.g. No disability, Has disability, Prefer not to say): ").strip()
    work_authorization = input("Are you authorized to work in the US? (Yes/No): ").strip()
    sponsorship = input("Do you require sponsorship now or in the future? (Yes/No): ").strip()
    visa_type = input("Visa type if applicable (e.g. H1B, F1 OPT, Green Card, Citizen, N/A): ").strip()

    info_text = f"""
Gender: {gender or 'Prefer not to say'}
Race/Ethnicity: {race or 'Prefer not to say'}
Veteran Status: {veteran_status or 'Prefer not to say'}
Disability Status: {disability_status or 'Prefer not to say'}
Work Authorization: {work_authorization or 'Yes'}
Requires Sponsorship: {sponsorship or 'No'}
Visa Type: {visa_type or 'N/A'}
""".strip()

    additional_collection.upsert(
        documents=[info_text],
        ids=["user_info"],
        metadatas=[{"type": "additional_info"}]
    )
    print("✅ Additional info stored in ChromaDB!")
    return info_text

def get_additional_info():
    """Fetch additional user info from ChromaDB."""
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    results = additional_collection.get(ids=["user_info"])
    if results["documents"]:
        return results["documents"][0]
    return None

### Agent handling short-answer questions field

In [17]:
# Handles short answer fields by trying to generate an answer from the resume.
# If the resume doesn't have enough info, it asks the user for more info.
# If company research is needed, it searches the web.
# Returns the final answer to fill in the field.
def short_answer_field(field_name: str, resume_text: str, job_url: str) -> str:
    print(f"\n📝 Short answer field detected: '{field_name}'")

    # Step 1: Try to generate answer from resume alone
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[
            f"Based on this resume, generate a concise answer for the job application field: '{field_name}'.\n"
            f"Resume:\n{resume_text}\n\n"
            f"If the resume does not have enough info to answer this field, respond with exactly: NEEDS_MORE_INFO\n"
            f"Otherwise respond with only the answer, nothing else."
        ]
    )
    answer = response.text.strip()

    # Step 2: If not enough info, ask user
    if "NEEDS_MORE_INFO" in answer.upper():
        print(f"⚠️ Resume doesn't have enough info for '{field_name}'.")
        user_info = input(f"Please provide info for '{field_name}' (or press Enter to skip): ").strip()

        if not user_info:
            print(f"⏭️ Skipping '{field_name}'")
            return None

        # Check if info is too vague
        vague_check = client_genai.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                f"Is this answer too vague to fill in a job application field called '{field_name}'? "
                f"Answer only YES or NO.\nAnswer: {user_info}"
            ]
        )
        while "YES" in vague_check.text.upper():
            print("⚠️ That answer is too vague.")
            user_info = input(f"Please provide a more specific answer for '{field_name}': ").strip()
            if not user_info:
                return None
            vague_check = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"Is this answer too vague to fill in a job application field called '{field_name}'? "
                    f"Answer only YES or NO.\nAnswer: {user_info}"
                ]
            )

        # Check if company research is needed
        research_check = client_genai.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                f"Does answering the job application field '{field_name}' require researching the company at {job_url}? "
                f"Answer only YES or NO."
            ]
        )

        extra_context = ""
        if "YES" in research_check.text.upper():
            print(f"🔍 Researching company for '{field_name}'...")
            research = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"Summarize relevant info about the company at {job_url} "
                    f"that would help answer the job application field: '{field_name}'."
                ]
            )
            extra_context = research.text.strip()
            print(f"✅ Company research done.")

        # Generate final answer with user info + extra context
        final_response = client_genai.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                f"Generate a concise job application answer for the field '{field_name}'.\n"
                f"Resume:\n{resume_text}\n"
                f"User provided: {user_info}\n"
                f"Extra context: {extra_context}\n"
                f"Return only the answer, nothing else."
            ]
        )
        answer = final_response.text.strip()

    return answer

### Phil fills in the application field

In [18]:
import json
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI

# Opens the job application in a new browser window.
# Waits for user to confirm they are signed in and have uploaded their resume.
# Uses additional info from ChromaDB for demographic and work authorization fields.
# Handles short answer fields using short_answer_field().
# Fills all remaining fields using the resume text.
# Saves to application_history.csv if all fields are filled.
async def fill_application(job_url: str, resume_text: str, additional_info: str = ""):
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_KEY"),
        model="openai/gpt-4o",
    )

    browser_session = BrowserSession(
        browser_profile=BrowserProfile(headless=False, keep_alive=True)
    )
    await browser_session.start()

    # Step 1: Open the browser to the job URL
    agent_open = Agent(
        task=f"Go to {job_url} and wait. Do not fill anything.",
        llm=llm,
        browser_session=browser_session,
        max_steps=3,
    )
    await agent_open.run()

    # Step 2: Wait for user to sign in
    signed_in = input("Are you signed in (if needed)? (y/n): ").strip().lower()
    while signed_in != "y" and signed_in != "yes":
        signed_in = input("Please sign in and then enter y to continue: ").strip().lower()

    # Step 3: Wait for user to upload resume manually
    input("Please upload your resume manually in the browser if required, then press Enter to continue...")

    # Step 4: Detect short answer fields before filling
    agent_detect = Agent(
        task=(
            f"The page {job_url} is already open. "
            "Look at the page and do the following:\n"
            "1. Find the company name and job title from the page.\n"
            "2. Find any SHORT ANSWER or ESSAY fields (e.g. 'Why do you want to work here?', 'Tell us about yourself', 'Cover letter', etc.).\n"
            "Return ONLY a JSON like this:\n"
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": ["field1", "field2"]}'
            "\nIf there are no short answer fields, return: "
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": []}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=5,
        max_actions_per_step=3,
    )
    detect_result = await agent_detect.run()
    detect_final = detect_result.final_result()

    # Step 5: Handle short answer fields
    short_answer_context = ""
    try:
        detected = json.loads(detect_final)
        short_answer_fields = detected.get("short_answer_fields", [])
        detected_company = detected.get("company_name", "Unknown")
        detected_job_title = detected.get("job_title", "Unknown")
        print(f"🏢 Company: {detected_company} | 💼 Job Title: {detected_job_title}")
        if short_answer_fields:
            print(f"\n📋 Short answer fields found: {short_answer_fields}")
            answers = {}
            for field in short_answer_fields:
                answer = short_answer_field(field, resume_text, job_url)
                if answer:
                    answers[field] = answer
            if answers:
                short_answer_context = "\n".join([f"{k}: {v}" for k, v in answers.items()])
    except Exception:
        print("⚠️ Could not detect short answer fields, skipping.")

    # Step 6: Fill all fields
    agent_fill = Agent(
        task=(
            f"The page {job_url} is already open. "
            f"Use this resume to fill out the application:\n{resume_text}\n\n"
            + (f"Use this additional user info for demographic and work authorization fields:\n{additional_info}\n\n" if additional_info else "")
            + (f"Use these pre-generated answers for short answer fields:\n{short_answer_context}\n\n" if short_answer_context else "")
            +
            "Instructions:\n"
            "1. Find every empty field on the page.\n"
            "2. If the field can be answered from the resume, fill it in.\n"
            "3. For DROPDOWN fields:\n"
            "   - Click the dropdown to open it.\n"
            "   - Immediately click on one of the visible options — do NOT scroll, do NOT search, do NOT re-click the dropdown.\n"
            "   - After clicking an option, move on to the next field immediately.\n"
            "   - If after 2 attempts the dropdown still does not respond, use JavaScript to select the value directly and move on.\n"
            "   - NEVER attempt the same dropdown more than 2 times.\n"
            "4. For SENSITIVE fields (Gender, Race, Veteran status, Disability status, Ethnicity):\n"
            "   - Use the additional user info provided above to fill these fields.\n"
            "   - If not found in additional info, pick the FIRST available option.\n"
            "5. For SPONSORSHIP/WORK AUTHORIZATION fields:\n"
            "   - Use the work authorization and sponsorship info from the additional user info above.\n"
            "   - If unclear, select 'Yes' for authorization and 'No' for sponsorship as default.\n"
            "6. For COUNTRY fields: select 'United States' unless the resume says otherwise.\n"
            "7. For FILE UPLOAD fields (resume, CV): skip them — the user has already uploaded manually.\n"
            "8. For other FILE UPLOAD fields (cover letter, portfolio, other documents): skip them.\n"
            "9. For SHORT ANSWER or ESSAY fields: use the pre-generated answers provided above.\n"
            "10. Do NOT click submit or login.\n"
            "11. When all fields are processed, return ONLY a JSON like this:\n"
            '{"filled": ["field1", "field2"], "empty": ["field3"], "skipped_uploads": ["resume", "cover letter"], "company_name": "Google", "job_title": "Software Engineer"}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=50,
        max_actions_per_step=10,
        include_attributes=["id", "name", "type", "placeholder", "aria-label", "label", "role", "data-testid"],
    )

    result = await agent_fill.run()
    final = result.final_result()
    print("Agent result:", final)

    try:
        parsed = json.loads(final)
        empty_fields = parsed.get("empty", [])
        company_name = parsed.get("company_name", detected_company)
        job_title = parsed.get("job_title", detected_job_title)

        if empty_fields:
            print(f"⚠️ Empty fields remaining: {empty_fields}")
        else:
            print("✅ All fields filled!")
            save_application_history(
                url=job_url,
                company_name=company_name,
                time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
                job_title=job_title,
            )
    except Exception:
        if final and "empty" not in final.lower():
            save_application_history(
                url=job_url,
                company_name=detected_company,
                time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
                job_title=detected_job_title,
            )
        else:
            print("⚠️ Could not confirm all fields filled.")

    return final

### Agent will first ask user if they would like to upload a new resume if it's their first time. Otherwise, it will grab the resume that's stored in ChromaDB and use that to fill in the field

In [ ]:
# async def apply_to_job_with_agent():
#     resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")
    
#     while(resume_exists != "y" and resume_exists != "n"):
#         resume_exists = input("Invalid input. Please enter y or n:")
    
#     if resume_exists == "y":
#         file_path = input("Please enter the path to your resume PDF: ").strip()
#         extracted_info = parse_resume(file_path)
#         verified_info = verify_resume_info(extracted_info)
#         store_resume_in_chromadb(verified_info, file_path)
#         resume_text = verified_info
#     else:
#         print("Using existing resume data. Moving on to job application...")
#         results = collection.get(limit=1, include=["documents"])
#         if not results["documents"]:
#             raise RuntimeError("No resume found in ChromaDB. Please upload a resume first.")
#         resume_text = results["documents"][0]
#         print("✅ Resume fetched from ChromaDB")

#     url = input("Please enter the URL of the job application you are applying to: ").strip()
#     await fill_application(url, resume_text)

# await apply_to_job_with_agent()

# Main function — asks user if they have a new resume, fetches from ChromaDB if not,
# collects or fetches additional demographic/work authorization info,
# then prompts for the job URL and kicks off the application agent.
async def apply_to_job_with_agent():
    resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")
    while resume_exists != "y" and resume_exists != "n":
        resume_exists = input("Invalid input. Please enter y or n: ")

    if resume_exists == "y":
        file_path = input("Please enter the path to your resume PDF: ").strip()
        extracted_info = parse_resume(file_path)
        verified_info = verify_resume_info(extracted_info)
        store_resume_in_chromadb(verified_info, file_path)
        resume_text = verified_info
    else:
        print("Using existing resume data. Moving on to job application...")
        results = collection.get(limit=1, include=["documents"])
        if not results["documents"]:
            raise RuntimeError("No resume found in ChromaDB. Please upload a resume first.")
        resume_text = results["documents"][0]
        print("✅ Resume fetched from ChromaDB")

    # Collect or fetch additional info
    additional_info = get_additional_info()
    if not additional_info:
        print("\n📋 No additional info found. Let's collect it now.")
        additional_info = collect_and_store_additional_info()
    else:
        update = input("\nWould you like to update your demographic/work authorization info? (y/n): ").strip().lower()
        if update == "y":
            additional_info = collect_and_store_additional_info()

    url = input("Please enter the URL of the job application you are applying to: ").strip()
    await fill_application(url, resume_text, additional_info)

await apply_to_job_with_agent()


Using existing resume data. Moving on to job application...
✅ Resume fetched from ChromaDB

📋 No additional info found. Let's collect it now.

📋 Please provide the following additional info (press Enter to skip any field):
✅ Additional info stored in ChromaDB!


INFO     [Agent] 🔗 Found URL in task: https://job-boards.greenhouse.io/attentive/jobs/4209296009, adding as initial action...
INFO     [Agent] 🎯 Task: Go to https://job-boards.greenhouse.io/attentive/jobs/4209296009 and wait. Do not fill anything.
INFO     [Agent] Starting a browser-use agent with version 0.12.6, with provider=openai and model=openai/gpt-4o
INFO     [Agent]   ▶️   navigate: url: https://job-boards.greenhouse.io/attentive/jobs/4209296009, new_tab: False
INFO     [tools] 🔗 Navigated to https://job-boards.greenhouse.io/attentive/jobs/4209296009
INFO     [Agent] 

INFO     [Agent] 📍 Step 1:
INFO     [Agent]   👍 Eval: Successfully navigated to the specified job listing page and waited as instructed. Verdict: Success
INFO     [Agent]   🧠 Memory: Navigated to the AI Intern job listing page at Attentive and confirmed the page is loaded. No further interaction was requested.
INFO     [Agent]   🎯 Next goal: No further action is required as per the user's request.
INFO     [Agent

🏢 Company: Attentive | 💼 Job Title: AI Intern

📋 Short answer fields found: ["If 'YES', what is the basis of your current employment authorization? And when does your current employment authorization expire? If H-1B, is there an approved I-140?"]

📝 Short answer field detected: 'If 'YES', what is the basis of your current employment authorization? And when does your current employment authorization expire? If H-1B, is there an approved I-140?'
⚠️ Resume doesn't have enough info for 'If 'YES', what is the basis of your current employment authorization? And when does your current employment authorization expire? If H-1B, is there an approved I-140?'.


INFO     [Agent] 🎯 Task: The page https://job-boards.greenhouse.io/attentive/jobs/4209296009 is already open. Use this resume to fill out the application:
Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
    



⏸️ Paused the agent and left the browser open.
	Press [Enter] to resume or [Ctrl+C] again to quit.


WARNING  [BrowserSession] 🔄 WebSocket reconnection attempt 2/3...
WARNING  [BrowserSession] 🔄 Reconnection attempt 2 failed: ConnectionRefusedError: [Errno 61] Connect call failed ('127.0.0.1', 52250)
WARNING  [BrowserSession] 🔄 WebSocket reconnection attempt 3/3...
WARNING  [BrowserSession] 🔄 Reconnection attempt 3 failed: ConnectionRefusedError: [Errno 61] Connect call failed ('127.0.0.1', 52250)
ERROR    [BrowserSession] 🔄 All 3 reconnection attempts failed
WARNING  [Agent] The agent was interrupted mid-step


## PLANS FOR NEXT MEETING:


## Plans for spring break:
* Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* Implementing a loop for our agent to handle when it comes to short-answer response
  * Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
  * Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
  * Have LLM generate a short answer response and fill out the field with Skyvern
* IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve